# Step 8 — XAI Method Comparison

Comparison of 5 explainability methods on the SMOTE + LogisticRegression model:
1. **Permutation Feature Importance (PFI)**
2. **SHAP values** (Shapley Additive Explanations)
3. **Model Coefficients**
4. **LIME** (Local Interpretable Model-agnostic Explanations)
5. **Feature Ablation**

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from asd_pipeline_utils import (
    RANDOM_STATE, SMOTE_PFI_PATH, XAI_COMPARISON_PATH,
    get_targets, load_analysis_frame, split_features_and_metadata,
    run_xai_comparison,
)

final_df = load_analysis_frame()
X, meta = split_features_and_metadata(final_df)
_, y_multi = get_targets(meta)

# Use top 50 PFI genes from Step 6
pfi_df = pd.read_csv(SMOTE_PFI_PATH)
xai_genes = pfi_df.head(50)["gene"].tolist()
X_xai = X[xai_genes].copy()

# Preprocess and train
imp = SimpleImputer(strategy="median")
sc = StandardScaler()
X_imp = pd.DataFrame(imp.fit_transform(X_xai), index=X_xai.index, columns=X_xai.columns)
X_sc = pd.DataFrame(sc.fit_transform(X_imp), index=X_imp.index, columns=X_imp.columns)

sm = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = sm.fit_resample(X_sc, y_multi)

clf = LogisticRegression(solver="saga", max_iter=8000, random_state=RANDOM_STATE)
clf.fit(X_res, y_res)

print(f"Step 8: XAI comparison on {len(xai_genes)} genes (5 methods)\n")

# Run all 5 XAI methods
rankings, top_sets, consensus = run_xai_comparison(clf, X_sc, y_multi, xai_genes)

top_k = 20
print(f"=== Top {top_k} genes by each XAI method ===\n")
table = pd.DataFrame({
    "Rank": range(1, top_k + 1),
    **{name: df.head(top_k)["gene"].values for name, df in rankings.items()}
})
print(table.to_string(index=False))

# Overlap
from itertools import combinations
print(f"\n=== Overlap Analysis (top {top_k}) ===")
print(f"All 5 methods agree: {len(consensus)} genes -> {sorted(consensus)}")
for (n1, s1), (n2, s2) in combinations(top_sets.items(), 2):
    print(f"  {n1} & {n2}: {len(s1 & s2)} genes")
print(f"Union: {len(set.union(*top_sets.values()))} genes")

# Save comparison
comparison_rows = []
for gene in set.union(*top_sets.values()):
    row = {"gene": gene, "methods_in_top20": sum(1 for s in top_sets.values() if gene in s)}
    comparison_rows.append(row)
comp_df = pd.DataFrame(comparison_rows).sort_values(["methods_in_top20", "gene"], ascending=[False, True])
comp_df.to_csv(XAI_COMPARISON_PATH, index=False)
print(f"\nSaved to {XAI_COMPARISON_PATH}")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 5, figsize=(28, 8))
colors = ["steelblue", "coral", "seagreen", "goldenrod", "mediumpurple"]

for ax, (name, df), color in zip(axes, rankings.items(), colors):
    top = df.head(top_k)
    ax.barh(range(top_k), top["score"].values, color=color)
    ax.set_yticks(range(top_k))
    ax.set_yticklabels(top["gene"].values, fontsize=7)
    ax.invert_yaxis()
    ax.set_title(f"{name} - Top {top_k}")

plt.suptitle("XAI Method Comparison - Top 20 Genes (5 Methods)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("xai_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: xai_comparison.png")